In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import torch
from nanodrz.model import DiarizeGPT, Config
from nanodrz import data
from nanodrz.data import artificial_diarisation_sample
from nanodrz.utils import visualise_annotation, play
from nanodrz.download import dl_scp_file
from nanodrz import format_conversions as format

device = torch.device("cuda:1")
torch.cuda.set_device(device)
ckpt = torch.load("/home/harry/runs/nanodrz/1708087744/0014000.pt", map_location=device)

In [ ]:
model:DiarizeGPT = DiarizeGPT.from_pretrained(ckpt).cuda()

In [ ]:
# Use the same parameters that the model was trained on to generate a sample
audio, labels = artificial_diarisation_sample(data.libritts_dev(), min_secs=20, num_speakers=2, max_secs=30)
reference = format.labels_to_annotation(labels)

nlabels = model.generate(audio.cuda(), temperature=1, max_steps=(len(labels))*3)

hypothesis = format.labels_to_annotation(nlabels)
visualise_annotation(labels)
visualise_annotation(nlabels)

from pyannote.core import Annotation
from pyannote.metrics.diarization import DiarizationErrorRate

metric = DiarizationErrorRate()
metric(reference, hypothesis)